# Python Libraries

In [35]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

# Configuration Classes for hyperparameters

In [36]:
@dataclass
class EnvironmentConfig:
    grid_size: int = 5
    goal: tuple[int, int] = (grid_size-1, grid_size-1)

@dataclass
class RewardConfig:
    step_reward: int = -1
    invalid_move_penalty: int = -2
    pickup_reward: int = 20
    delivery_reward: int = 100

@dataclass
class QLearningConfig:
    alpha: float = 0.10
    gamma: float = 0.95
    epsilon: float = 1.00
    epsilon_decay: float = 0.995
    min_epsilon: float = 0.05

@dataclass
class TrainingConfig:
    episodes: int = 5000
    max_steps: int = 100

#actions
ACTIONS = [
    (-1, 0),   # Up
    (1, 0),    # Down
    (0, -1),   # Left
    (0, 1),    # Right
    (-1, -1),  # Up Left
    (-1, 1),   # Up Right
    (1, -1),   # Down Left
    (1, 1)     # Down Right
]

# Configuration Objects

In [37]:
environment = EnvironmentConfig()
rewards = RewardConfig()
q_learning = QLearningConfig()
training = TrainingConfig()

# Environment

### Tasks complete:
1. New environment - __init__
2. Agent new position - reset() - random_position()
3. New pickup position - reset() - random_pickup()
4. Reward for every steps - step()
5. Vitualize environment - render()

### Worflow
picture link: https://prnt.sc/dP1kype0l-rg


In [33]:
class GridWorld:
    # Initialize the GridWorld environment
    def __init__(self):
        # Size of the square grid
        self.size = environment.grid_size

        # Fixed delivery destination (bottom-right corner)
        self.goal = (
            self.size - 1,
            self.size - 1
        )

    # Reset the environment for a new episode
    def reset(self):
        # Place the agent at a random location
        self.agent = self.random_position()

        # Place the pickup item at a valid location
        self.pickup = self.random_pickup()

        # Agent starts without carrying the item
        self.carrying = False

        # Return the initial state
        return self.get_state()

    # Generate a random position inside the grid
    def random_position(self):
        row = np.random.randint(self.size)
        col = np.random.randint(self.size)
        return (row, col)

    # Generate a pickup position that is not occupied
    # by the agent or the goal
    def random_pickup(self):
        pickup = self.random_position()

        # Keep generating until a valid position is found
        while self.valid_pickup(pickup) is False:
            pickup = self.random_position()

        return pickup

    # Check whether the pickup position is valid
    def valid_pickup(self, pickup):
        # Pickup cannot overlap with the agent
        different_agent = pickup != self.agent

        # Pickup cannot overlap with the goal
        different_goal = pickup != self.goal

        return different_agent and different_goal

    # Return the current environment state
    def get_state(self):
        return (
            self.agent,
            self.pickup,
            self.carrying
        )

    # Execute one environment step
    def step(self, action):
        # Base reward received every step
        reward = rewards.step_reward

        # Apply movement and invalid move penalty if needed
        reward += self.move(action)

        # Apply pickup reward if the item is collected
        reward += self.collect_reward()

        # Check whether delivery is complete
        completed = self.complete_delivery()

        # Give delivery reward if the episode is finished
        if completed:
            reward += rewards.delivery_reward

        # Return next state, reward and termination flag
        return (
            self.get_state(),
            reward,
            completed
        )

    # Move the agent according to the selected action
    def move(self, action):
        # Convert action index into row/column movement
        dx, dy = ACTIONS[action]

        # Compute the next position
        row = self.agent[0] + dx
        col = self.agent[1] + dy

        # Check whether the move stays inside the grid
        inside = self.inside_grid(row, col)

        if inside:
            # Update the agent position
            self.agent = (row, col)
            return 0

        # Penalize invalid moves outside the grid
        return rewards.invalid_move_penalty

    # Check whether a position is inside the grid boundaries
    def inside_grid(self, row, col):
        valid_row = 0 <= row < self.size
        valid_col = 0 <= col < self.size
        return valid_row and valid_col

    # Check whether the agent reaches the pickup location
    def collect_reward(self):
        pickup_cell = self.agent == self.pickup

        if pickup_cell:
            return self.pickup_item()

        return 0

    # Pick up the item if it has not already been collected
    def pickup_item(self):
        carrying_item = self.carrying

        # Prevent collecting the same item multiple times
        if carrying_item:
            return 0

        # Mark the item as collected
        self.carrying = True

        # Give pickup reward
        return rewards.pickup_reward

    # Check whether the delivery task is complete
    def complete_delivery(self):
        # Agent has reached the goal
        at_goal = self.agent == self.goal

        # Agent is carrying the pickup item
        carrying_item = self.carrying

        return at_goal and carrying_item

    # Display the current environment
    def render(self):
        # Create an empty grid
        grid = self.create_grid()

        # Draw pickup, goal and agent
        self.place_pickup(grid)
        self.place_goal(grid)
        self.place_agent(grid)

        # Print the grid to the console
        self.print_grid(grid)

    # Create an empty grid filled with '.'
    def create_grid(self):
        return [
            ['.' for _ in range(self.size)]
            for _ in range(self.size)
        ]

    # Place the pickup item on the grid
    def place_pickup(self, grid):
        # Hide the pickup after it has been collected
        if self.carrying:
            return

        row, col = self.pickup
        grid[row][col] = "A"

    # Place the delivery goal on the grid
    def place_goal(self, grid):
        row, col = self.goal
        grid[row][col] = "B"

    # Place the agent on the grid
    def place_agent(self, grid):
        row, col = self.agent

        # Show a different symbol when carrying the item
        if self.carrying:
            grid[row][col] = "P*"
            return

        grid[row][col] = "P"

    # Print the grid in a formatted layout
    def print_grid(self, grid):
        print("-" * (self.size * 4))

        for row in grid:
            print(" ".join(f"{cell:>3}" for cell in row))

        print("-" * (self.size * 4))

# Testing Environment

In [34]:
# Create the environment
env = GridWorld()

# Start a new episode
state = env.reset()

print("Initial State:")
print(state)

# Display the environment
env.render()

# Test every action once
for action in range(8):
    print(f"\nAction: {action}")

    next_state, reward, done = env.step(action)

    print("State :", next_state)
    print("Reward:", reward)
    print("Done  :", done)

    env.render()

    if done:
        print("Episode Finished!")
        break

Initial State:
((4, 3), (3, 1), False)
--------------------
  .   .   .   .   .
  .   .   .   .   .
  .   .   .   .   .
  .   A   .   .   .
  .   .   .   P   B
--------------------

Action: 0
State : ((3, 3), (3, 1), False)
Reward: -1
Done  : False
--------------------
  .   .   .   .   .
  .   .   .   .   .
  .   .   .   .   .
  .   A   .   P   .
  .   .   .   .   B
--------------------

Action: 1
State : ((4, 3), (3, 1), False)
Reward: -1
Done  : False
--------------------
  .   .   .   .   .
  .   .   .   .   .
  .   .   .   .   .
  .   A   .   .   .
  .   .   .   P   B
--------------------

Action: 2
State : ((4, 2), (3, 1), False)
Reward: -1
Done  : False
--------------------
  .   .   .   .   .
  .   .   .   .   .
  .   .   .   .   .
  .   A   .   .   .
  .   .   P   .   B
--------------------

Action: 3
State : ((4, 3), (3, 1), False)
Reward: -1
Done  : False
--------------------
  .   .   .   .   .
  .   .   .   .   .
  .   .   .   .   .
  .   A   .   .   .
  .   .   .   P   B


# Q-Table

A Dictionary: \
&nbsp;&nbsp;&nbsp;&nbsp;Key: agent, pickup, carry \
&nbsp;&nbsp;&nbsp;&nbsp;value: up, down, left, right, up-left, up-right, down-left, down-right

In [38]:
# Initialize the Q-table
q_table = {}

def initialize_state(q_table, state):
    if state in q_table:
        return
    q_table[state] = np.zeros(len(ACTIONS))  #initial val 0

# ε-greedy

#### Work does
Select an action using the epsilon-greedy policy.

    Parameters:
        q_table : Dictionary containing Q-values.
        state   : agent, pickup, carry
        epsilon : Probability of choosing a random action.

    Returns:
        action (int): An action index from 0 to 7.

In [39]:
def epsilon_greedy(q_table, state, epsilon):
    # Make sure the current state exists in the Q-table
    initialize_state(q_table, state)

    # Explore: choose a random action
    if np.random.random() < epsilon:
        action = np.random.randint(len(ACTIONS))
    # Exploit: choose the action with the highest Q-value
    else:
        action = np.argmax(q_table[state])

    return action